In [1]:
# PART 0 — Imports and file paths
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import t

DATA_DIR = Path("Data")
OUTPUT_DIR = Path("results")
REQUIRED_FILES = [
    "quality_samples.csv",
    "supplier_offers_SIMULATED.csv",
    "sourcing_scenarios_SIMULATED.csv",
]

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Working directory:", Path.cwd())

for filename in REQUIRED_FILES:
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(
            f"Missing {DATA_DIR / filename}. Open Jupyter in the project root."
        )


Python: 3.14.6
pandas: 3.0.6
NumPy: 2.5.3
Matplotlib: 3.11.2
Working directory: C:\Users\Administrator


FileNotFoundError: Missing Data\quality_samples.csv. Open Jupyter in the project root.

In [ ]:
# Read the supplied files and check for missing or inconsistent data
quality = pd.read_csv(
    DATA_DIR / REQUIRED_FILES[0],
    dtype={"sample_id": "string", "supplier_id": "string"},
)
offers = pd.read_csv(
    DATA_DIR / REQUIRED_FILES[1], dtype={"supplier_id": "string"}
)
scenarios = pd.read_csv(
    DATA_DIR / REQUIRED_FILES[2],
    dtype={"scenario": "string", "unavailable_supplier": "string"},
)
quality_samples = quality  # Existing group notebook compatibility.

required_columns = {
    "quality": {"sample_id", "supplier_id", "origin", "processing_method",
                "cup_points", "aroma", "flavour"},
    "offers": {"supplier_id", "fictional_name", "price_gbp_per_kg",
               "capacity_kg", "minimum_order_kg", "lead_time_days",
               "on_time_percent"},
    "scenarios": {"scenario", "demand_kg", "budget_gbp",
                  "minimum_weighted_quality", "deadline_days",
                  "maximum_supplier_share", "unavailable_supplier"},
}

for name, table in [("quality", quality), ("offers", offers),
                    ("scenarios", scenarios)]:
    missing_cols = required_columns[name] - set(table.columns)
    if missing_cols:
        raise ValueError(f"{name} is missing columns: {sorted(missing_cols)}")
    print(f"\n{name.upper()} — {table.shape[0]} rows, {table.shape[1]} columns")
    print("Missing by column:\n", table.isna().sum().to_string())
    print("Exact duplicate rows:", int(table.duplicated().sum()))

assert quality["sample_id"].notna().all() and quality["sample_id"].is_unique
assert quality["supplier_id"].notna().all()
assert quality["origin"].notna().all()
assert offers["supplier_id"].notna().all() and offers["supplier_id"].is_unique
assert scenarios["scenario"].notna().all() and scenarios["scenario"].is_unique
assert not quality.duplicated().any()
assert not offers.duplicated().any()
assert set(quality["supplier_id"]) == set(offers["supplier_id"])
assert set(scenarios["scenario"]) == {"Base", "Demand increase", "S01 unavailable"}
assert set(scenarios["unavailable_supplier"].dropna().astype(str)) <= set(offers["supplier_id"]) | {"None"}

# Audit numeric fields, without deleting historical samples.
for col, lower, upper in [("cup_points", 0, 100),
                          ("aroma", 0, 10), ("flavour", 0, 10)]:
    values = pd.to_numeric(quality[col], errors="coerce")
    assert values.notna().all(), f"Non-numeric/missing values in {col}"
    assert np.isfinite(values.to_numpy(dtype=float)).all(), col
    assert values.between(lower, upper).all(), f"Out-of-range {col}"
    if col == "cup_points":
        assert values.gt(0).all(), "Source requires positive cup scores"
    print(f"{col} range: {values.min():.2f} – {values.max():.2f}")

# processing_method is descriptive, not part of the required quality score.
missing_processing = (
    quality["processing_method"].isna()
    | quality["processing_method"].astype("string").str.strip().eq("").fillna(False)
)
print("Missing/blank processing methods:", int(missing_processing.sum()))

# Validate simulated numeric restrictions before relying on them.
offer_numbers = ["price_gbp_per_kg", "capacity_kg", "minimum_order_kg",
                 "lead_time_days", "on_time_percent"]
scenario_numbers = ["demand_kg", "budget_gbp", "minimum_weighted_quality",
                    "deadline_days", "maximum_supplier_share"]
assert offers[offer_numbers].notna().all().all()
assert scenarios[scenario_numbers].notna().all().all()
assert offers["price_gbp_per_kg"].gt(0).all()
assert offers["capacity_kg"].gt(0).all()
assert offers["minimum_order_kg"].ge(0).all()
assert offers["minimum_order_kg"].le(offers["capacity_kg"]).all()
assert offers["lead_time_days"].ge(0).all()
assert offers["on_time_percent"].between(0, 100).all()
assert scenarios["demand_kg"].gt(0).all()
assert scenarios["budget_gbp"].gt(0).all()
assert scenarios["deadline_days"].gt(0).all()
assert scenarios["minimum_weighted_quality"].between(0, 100).all()
assert scenarios["maximum_supplier_share"].between(0, 1).all()
assert (offers[["capacity_kg", "minimum_order_kg", "lead_time_days"]] % 1 == 0).all().all()
assert (scenarios[["demand_kg", "deadline_days"]] % 1 == 0).all().all()

# Summarise the historical quality scores by supplier
origin_per_bidder = quality.groupby("supplier_id")["origin"].nunique(dropna=False)
assert origin_per_bidder.eq(1).all(), "A bidder has multiple origin labels"

quality_tagged = quality.assign(processing_missing=missing_processing.astype(int))
quality_summary = (
    quality_tagged.groupby("supplier_id", as_index=False)
    .agg(
        n=("cup_points", "count"),
        mean_quality=("cup_points", "mean"),
        sd_quality=("cup_points", "std"),  # Sample SD (ddof=1).
        origin=("origin", "first"),
        missing_processing=("processing_missing", "sum"),
        mean_aroma=("aroma", "mean"),
        mean_flavour=("flavour", "mean"),
    )
)
assert quality_summary["n"].ge(2).all(), "t CI requires n >= 2"
assert quality_summary["n"].sum() == len(quality)

# Two-sided 95% Student-t confidence interval for each bidder mean.
quality_summary["se_quality"] = (
    quality_summary["sd_quality"] / np.sqrt(quality_summary["n"])
)
quality_summary["t_critical"] = t.ppf(0.975, quality_summary["n"] - 1)
quality_summary["ci95_margin"] = (
    quality_summary["t_critical"] * quality_summary["se_quality"]
)
quality_summary["ci95_low"] = (
    quality_summary["mean_quality"] - quality_summary["ci95_margin"]
)
quality_summary["ci95_high"] = (
    quality_summary["mean_quality"] + quality_summary["ci95_margin"]
)

# Weighted reconciliation: groups must reconstruct the original overall mean.
weighted_overall = (
    (quality_summary["n"] * quality_summary["mean_quality"]).sum()
    / quality_summary["n"].sum()
)
assert np.isclose(weighted_overall, quality["cup_points"].mean())

# Display the supplier-level quality statistics calculated above.
display(
    quality_summary[[
        "supplier_id", "n", "mean_quality", "sd_quality",
        "ci95_low", "ci95_high"
    ]].round(3)
)

# Add the supplier offers and check delivery eligibility
bidders = offers.merge(
    quality_summary, on="supplier_id", how="left",
    validate="one_to_one", indicator=True,
)
print("\nMerge results:\n", bidders["_merge"].value_counts().to_string())
assert len(bidders) == len(offers)
assert bidders["_merge"].eq("both").all()
bidders = bidders.drop(columns="_merge")

base_scen = scenarios.loc[scenarios["scenario"].eq("Base")].iloc[0]
bidders["eligible"] = bidders["lead_time_days"].le(
    int(base_scen["deadline_days"])
)
print("\nSupplier data for other group sections:")
print(bidders[["supplier_id", "n", "mean_quality", "sd_quality",
               "ci95_low", "ci95_high", "price_gbp_per_kg", "capacity_kg",
               "minimum_order_kg", "lead_time_days", "eligible"]]
      .sort_values("supplier_id").round(3).to_string(index=False))

# Plot the quality estimates and compare price against quality
ordered = bidders.sort_values("mean_quality", ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
x_positions = np.arange(len(ordered))
ax.errorbar(
    x_positions, ordered["mean_quality"],
    yerr=ordered["ci95_margin"], fmt="o", capsize=4,
    label="Mean ± 95% t CI",
)
quality_floor = float(base_scen["minimum_weighted_quality"])
ax.axhline(quality_floor, color="gray", linestyle="--",
           label="Portfolio weighted-quality minimum (not per-bidder)")
ax.set_xticks(x_positions, ordered["supplier_id"])
ax.set_xlabel("Bidder ID")
ax.set_ylabel("Cup score")
ax.set_title("Bidder mean quality with 95% confidence intervals")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for eligibility, marker in [(True, "o"), (False, "x")]:
    group = bidders.loc[bidders["eligible"].eq(eligibility)]
    ax.scatter(
        group["price_gbp_per_kg"], group["mean_quality"],
        s=40 + group["n"] * 5, marker=marker,
        label="Deadline-eligible" if eligibility else "Late (>28 days)",
    )
for row in bidders.itertuples():
    ax.annotate(row.supplier_id,
                (row.price_gbp_per_kg, row.mean_quality),
                xytext=(4, 3), textcoords="offset points")
ax.axhline(quality_floor, color="gray", linestyle="--")
ax.set_xlabel("Simulated offer price (£/kg)")
ax.set_ylabel("Historical mean cup quality")
ax.set_title("Bidder price–quality comparison (marker size = n)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

OUTPUT_DIR.mkdir(exist_ok=True)
bidders.to_csv(OUTPUT_DIR / "bidders_quality_summary.csv", index=False)
print("\nPart 1 finished: bidders table has", len(bidders), "rows")
print("All historical quality samples retained:", len(quality))
print("Missing processing methods:", int(missing_processing.sum()))
